In [ ]:
import json
import io
import pickle
from pathlib import Path

from PIL import Image
import ipywidgets as widgets
from IPython.display import display

# --- 1. Configuration ---
COMPONENT = "clarifier"
ARCHITECTURE = "convnext_tiny"      # must match save_name used in save_result()
TEST_SET_ID = "Atlantis"     # must match the TEST_SET_ID printed when wwtw_utils was imported

RESULTS_DIR = Path("../results")
RESULTS_FILE = RESULTS_DIR / f"results_{COMPONENT}_{ARCHITECTURE}_{TEST_SET_ID}.pkl"

# Flags are keyed by component + test set (not architecture) — a ground
# truth mistake is a property of the image, not of which model happened to
# get it wrong, so flags accumulate into one file even if you review
# several architectures' results against the same held-out facility.
FLAGS_FILE = Path(f"./relabel_flags_{COMPONENT}_{TEST_SET_ID}.json")

THUMB_SIZE = (160, 160)
BATCH_SIZE = 50
GRID_COLS = 5

# --- 2. Load results (no model, no inference) ---
if not RESULTS_FILE.exists():
    available = sorted(RESULTS_DIR.glob(f"results_{COMPONENT}_*.pkl"))
    raise FileNotFoundError(
        f"{RESULTS_FILE} not found. Available results files for component='{COMPONENT}':\n"
        + "\n".join(f"  {p.name}" for p in available)
    )

with open(RESULTS_FILE, "rb") as f:
    payload = pickle.load(f)

print(f"Loaded results: model={payload['model_name']} | component={payload['component']} | "
      f"test_set={payload.get('test_set_id', 'unknown')} | test_acc={payload['test_acc']:.3f}")

per_sample = payload["per_sample"]
misclassified = [item for item in per_sample if not item["correct"]]
print(f"Found {len(misclassified)} misclassified images (of {len(per_sample)} total test images).")

n_batches = max(1, -(-len(misclassified) // BATCH_SIZE))  # ceil div

# --- 3. Load Existing Flags ---
if FLAGS_FILE.exists():
    with open(FLAGS_FILE, "r") as f:
        flags = json.load(f)
else:
    flags = {}

def save_flags():
    with open(FLAGS_FILE, "w") as f:
        json.dump(flags, f, indent=2)

# --- 4. Widget State ---
state = {"batch_idx": 0}

def make_thumb_bytes(img_path):
    img = Image.open(img_path).convert("RGB")
    img.thumbnail(THUMB_SIZE)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()

def make_item_widget(item):
    path_str = item["path"]
    img_widget = widgets.Image(value=make_thumb_bytes(path_str), format="png",
                                width=THUMB_SIZE[0], height=THUMB_SIZE[1])

    label_html = widgets.HTML(
        f"<div style='font-size:11px'>"
        f"<b>True:</b> {item['true']}<br>"
        f"<b>Pred:</b> {item['pred']}<br>"
        f"<span style='color:gray'>{Path(path_str).name}</span>"
        f"</div>"
    )

    is_flagged = flags.get(path_str, {}).get("flagged", False)
    flag_btn = widgets.ToggleButton(
        value=is_flagged,
        description="Flagged" if is_flagged else "Flag for relabel",
        button_style="danger" if is_flagged else "",
        layout=widgets.Layout(width=f"{THUMB_SIZE[0]}px"),
    )

    def on_toggle(change, item=item, btn=flag_btn):
        path_str = item["path"]
        if change["new"]:
            flags[path_str] = {
                "true": item["true"], "pred": item["pred"], "flagged": True,
                "source_model": payload["model_name"], "test_set_id": payload.get("test_set_id"),
            }
            btn.description = "Flagged"
            btn.button_style = "danger"
        else:
            flags.pop(path_str, None)
            btn.description = "Flag for relabel"
            btn.button_style = ""
        save_flags()
        update_status()

    flag_btn.observe(on_toggle, names="value")

    return widgets.VBox(
        [img_widget, label_html, flag_btn],
        layout=widgets.Layout(border="1px solid #ddd", padding="4px", margin="2px", width=f"{THUMB_SIZE[0]+16}px")
    )

# --- 5. Rendering ---
output_area = widgets.Output()
status_label = widgets.Label()
batch_label = widgets.Label()

def update_status():
    status_label.value = f"Flagged for relabel: {len(flags)} / {len(misclassified)}"

def render_batch(idx):
    idx = max(0, min(idx, n_batches - 1))
    state["batch_idx"] = idx

    start = idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, len(misclassified))
    batch_items = misclassified[start:end]

    batch_label.value = f"Batch {idx + 1}/{n_batches}  (items {start + 1}-{end} of {len(misclassified)})"

    item_widgets = [make_item_widget(item) for item in batch_items]
    grid = widgets.GridBox(
        item_widgets,
        layout=widgets.Layout(grid_template_columns=f"repeat({GRID_COLS}, auto)")
    )

    with output_area:
        output_area.clear_output(wait=True)
        display(grid)

    update_status()

# --- 6. Controls ---
prev_btn = widgets.Button(description="⬅ Previous", button_style="")
next_btn = widgets.Button(description="Next ➡", button_style="")
export_btn = widgets.Button(description="Export flagged list (CSV)", button_style="info")
export_output = widgets.Output()

def on_prev(_):
    render_batch(state["batch_idx"] - 1)

def on_next(_):
    render_batch(state["batch_idx"] + 1)

def on_export(_):
    import csv
    out_path = Path(f"./flagged_for_relabel_{COMPONENT}_{TEST_SET_ID}.csv")
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["path", "true_label", "predicted_label", "source_model", "test_set_id"])
        for path_str, info in flags.items():
            writer.writerow([
                path_str, info["true"], info["pred"],
                info.get("source_model", ""), info.get("test_set_id", ""),
            ])
    with export_output:
        export_output.clear_output(wait=True)
        print(f"Exported {len(flags)} flagged items to {out_path}")

prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
export_btn.on_click(on_export)

controls = widgets.HBox([prev_btn, batch_label, next_btn])
top_bar = widgets.HBox([status_label, export_btn])

# --- 7. Display ---
display(widgets.VBox([top_bar, controls, output_area, export_output]))
render_batch(0)

Loaded results: model=ConvNeXt-Tiny | component=clarifier | test_set=Atlantis | test_acc=0.600
Found 36 misclassified images (of 90 total test images).
